# Föreläsning — Signal- och radiolära

Interaktiv version av Kapitel 1 och 2 från [index.md](./html/index.md). Kör cellerna med **Shift+Enter** och dra i slidrarna under föreläsningen.

**Beroenden:** `ipywidgets`, `numpy`, `matplotlib` — alla finns i [requirements.txt](../../requirements.txt).

## Kapitel 1 — Sinuskurvan

$y(t) = A \cdot \sin(2\pi f t + \phi)$

- **A** — amplitud (höjd)
- **f** — frekvens (svängningar per sekund, Hz)
- **φ** — fasförskjutning (var i cykeln vågen startar, i grader)

Den blå streckade kurvan är referensen $\sin(2\pi t)$ — en sinusvåg med A=1, f=1 Hz, φ=0°.

In [19]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

%matplotlib inline

def plot_sine(A=1.0, f=1.0, phi_deg=0.0):
    phi = np.deg2rad(phi_deg)
    t = np.linspace(0, 2, 500)
    y = A * np.sin(2 * np.pi * f * t + phi)
    y_ref = np.sin(2 * np.pi * 1 * t)

    fig = plt.figure(figsize=(9, 4))
    plt.plot(t, y, color="orange",
             label=fr"$y(t) = {A:.1f}\cdot\sin(2\pi\cdot{f:.1f}\,t + \phi)$, φ = {phi_deg:.0f}°")
    plt.plot(t, y_ref, color="blue", linestyle="--", alpha=0.6,
             label=r"Referens: $\sin(2\pi t)$")
    plt.title("Sinuskurvan")
    plt.xlabel("Tid (s)")
    plt.ylabel("y(t)")
    plt.axhline(0, color="black", linewidth=0.5)
    plt.ylim(-5.5, 5.5)
    plt.grid(alpha=0.3)
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

interact(
    plot_sine,
    A=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="Amp (A)"),
    f=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="Freq (Hz)"),
    phi_deg=FloatSlider(value=0, min=-180, max=180, step=15, description="Fas (°)"),
);

interactive(children=(FloatSlider(value=1.0, description='Amp (A)', max=5.0, min=0.1), FloatSlider(value=1.0, …

## Kapitel 2 — Tids- och frekvensdomän

Vi visar upp till fyra sinusvågor i båda domänerna samtidigt. Notera att **varje sinus i tidsdomänen blir en topp i frekvensdomänen** — det är så Fouriertransformen "plockar isär" en signal i sina frekvenskomponenter. Varje signal har sin egen färg så att du kan följa den från tidsdomänen ner till motsvarande topp i FFT-spektrumet.

Slidrarna styr antal signaler, deras amplituder ($A_1$–$A_4$) och frekvenser ($f_1$–$f_4$). Pröva t.ex. att sätta f₁=f₂ — då ser du två toppar smälta ihop till en högre. Eller dra ner en amplitud till 0 för att "släcka" en signal.

In [20]:
def plot_time_freq(num_signals=4,
                   A1=1.0, A2=2.0, A3=3.0, A4=4.0,
                   f1=5.0, f2=10.0, f3=15.0, f4=20.0):
    fs = 1000
    t = np.linspace(0, 1, fs, endpoint=False)
    frequencies = [f1, f2, f3, f4][:num_signals]
    amplitudes = [A1, A2, A3, A4][:num_signals]
    colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"][:num_signals]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6))

    half = fs // 2
    freq = np.fft.fftfreq(len(t), 1 / fs)

    for i, (A, f, c) in enumerate(zip(amplitudes, frequencies, colors), start=1):
        signal = A * np.sin(2 * np.pi * f * t)
        spectrum = np.fft.fft(signal)

        ax1.plot(t, signal, color=c,
                 label=f"Signal {i} (A={A:.1f}, f={f:.1f} Hz)")

        markerline, stemline, baseline = ax2.stem(
            freq[:half], 2 / fs * np.abs(spectrum[:half]),
            basefmt=" ",
        )
        plt.setp(stemline, color=c, linewidth=1.5)
        plt.setp(markerline, color=c, markersize=6)

    ax1.set_title(f"Tidsdomän: {num_signals} sinusvåg(or)")
    ax1.set_xlabel("Tid (s)")
    ax1.set_ylabel("Amplitud")
    ax1.set_xlim(0, 0.5)
    ax1.grid(alpha=0.3)
    ax1.legend(loc="upper right", fontsize=9)

    ax2.set_title("Frekvensdomän (FFT-magnitud)")
    ax2.set_xlabel("Frekvens (Hz)")
    ax2.set_ylabel("Amplitud")
    ax2.set_xlim(0, 30)
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
    plt.close(fig)

interact(
    plot_time_freq,
    num_signals=IntSlider(value=4, min=1, max=4, description="Antal signaler"),
    A1=FloatSlider(value=1.0, min=0, max=5, step=0.1, description="A₁"),
    A2=FloatSlider(value=2.0, min=0, max=5, step=0.1, description="A₂"),
    A3=FloatSlider(value=3.0, min=0, max=5, step=0.1, description="A₃"),
    A4=FloatSlider(value=4.0, min=0, max=5, step=0.1, description="A₄"),
    f1=FloatSlider(value=5.0, min=1, max=25, step=0.5, description="f₁ (Hz)"),
    f2=FloatSlider(value=10.0, min=1, max=25, step=0.5, description="f₂ (Hz)"),
    f3=FloatSlider(value=15.0, min=1, max=25, step=0.5, description="f₃ (Hz)"),
    f4=FloatSlider(value=20.0, min=1, max=25, step=0.5, description="f₄ (Hz)"),
);

interactive(children=(IntSlider(value=4, description='Antal signaler', max=4, min=1), FloatSlider(value=1.0, d…